In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").is_dir() and (candidate / "project.ipynb").is_file():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pyspark.sql import SparkSession
from src.dataOI import DataOI

SEED = 42

# True: read the raw CSV, then clean, engineer text, split, encode, and save that split.
# False: skip those steps and load the saved train, validation, and test frames from data/processed/split.
REBUILD = False

# `spark.driver.memory` is read only when the JVM starts. Restart the kernel after changing it.
spark = SparkSession.builder \
                            .master("local[*]")\
                            .appName("DrugReviewClassification")\
                            .config("spark.driver.memory","4g")\
                            .getOrCreate()
spark_context = spark.sparkContext
spark_context.setLogLevel("ERROR")
data_oi = DataOI()


In [ ]:
from src.cleaning import data_cleaning
from src.text_features import text_feature_engineering
from src.encoder import encoder
from src.split import split, weight

if REBUILD:
    df_raw = data_oi.load_raw_data()
    df_cleaned = data_cleaning(df_raw)
    df_cleaned.cache().count()
    df_engineered = text_feature_engineering(df_cleaned, True)
    df_engineered.cache().count()
    df_train, df_valid, df_test = split(df_engineered, SEED)
    df_train = weight(df_train)
    df_train, df_valid, df_test = encoder(df_train, df_valid, df_test)
    data_oi.save_split_data(df_train, df_valid, df_test)
    data_oi.unpersist(df_cleaned, df_engineered)
    del df_raw, df_cleaned, df_engineered
else:
    df_train, df_valid, df_test = data_oi.load_split_data()


In [ ]:
from src.embeddings import EmbeddingModel
from src.text_features import fit_phrases
# Key in train_registry / predict_registry. TF-IDF uses baseline_train / baseline_predict.
embedding_name = "glove"
phrases = fit_phrases(df_train) if embedding_name == "glove" else None
embedding_model = EmbeddingModel(df_train, df_valid, df_test, phrases)

print(f"Embedding Model `{embedding_name}` Training")
EmbeddingModel.train_registry[embedding_name](embedding_model)


In [ ]:
print(f"Embedding Model `{embedding_name}` Running")
with data_oi.scope():
    train_embedding, valid_embedding, test_embedding = EmbeddingModel.predict_registry[embedding_name](embedding_model)
    data_oi.save_embedding_data(
        embedding_name,
        train_embedding,
        valid_embedding,
        test_embedding,
    )
    data_oi.unpersist(train_embedding, valid_embedding, test_embedding)
for extra in (embedding_model._train_expanded, embedding_model._train_expanded_phrase):
    if extra is not None:
        data_oi.unpersist(extra)
data_oi.unpersist(df_train, df_valid, df_test)
del embedding_model, phrases
del train_embedding, valid_embedding, test_embedding
del df_train, df_valid, df_test


In [ ]:
from src.analytics import AnalyticsModel

train_embedding, valid_embedding, test_embedding = data_oi.load_embedding_data(embedding_name)
analytics_models = {}
analytics_models[embedding_name] = AnalyticsModel(
    embedding_name,
    train_embedding,
    valid_embedding,
    test_embedding,
)


In [ ]:
train_pred_dict = {}
valid_pred_dict = {}
test_pred_dict = {}
for name, function in AnalyticsModel.registry.items():
    print(f"Embedding Model `{embedding_name}` Analytics Model `{name}` Running")
    # Format: `{analytics}__{embedding}`. Either name may contain a single underscore.
    model_name = f"{name}__{embedding_name}"
    train_pred, valid_pred, test_pred = function(analytics_models[embedding_name])
    data_oi.save_prediction_data(model_name, train_pred, valid_pred, test_pred)
    analytics_models[embedding_name].release_features()
    train_pred, valid_pred, test_pred = train_pred.cache(), valid_pred.cache(), test_pred.cache()
    train_pred.count()
    valid_pred.count()
    test_pred.count()
    train_pred_dict[model_name] = train_pred
    valid_pred_dict[model_name] = valid_pred
    test_pred_dict[model_name] = test_pred

data_oi.unpersist(train_embedding, valid_embedding, test_embedding)
del train_embedding, valid_embedding, test_embedding, analytics_models


In [ ]:
from src.evaluation import Scorer

# Score only predictions produced by this experiment.
scorer_train = Scorer(train_pred_dict)
print("Training Data:")
scorer_train.show_metrics(truncate = 80)

scorer_valid = Scorer(valid_pred_dict)
print("Validation Data:")
scorer_valid.show_metrics(truncate = 80)

scorer_test = Scorer(test_pred_dict)
print("Testing Data:")
scorer_test.show_metrics(truncate = 80)